# Day 5 — Healthcare Dataset Cleaning and EDA

## Objective
Use NumPy and pandas to inspect, validate, clean, summarize and document a
public healthcare dataset.

## Dataset
Heart Failure Clinical Records — UCI Machine Learning Repository.

## Important Limitation
This notebook performs descriptive exploratory analysis only. It does not make
clinical recommendations or establish causal relationships.

In [1]:
# ----------------------------------------------------
# IMPORTS AND VERSION CHECK
# ----------------------------------------------------
from pathlib import Path  # Object-oriented filesystem path management
import json               # Used for saving the dataset schema as a JSON file
import numpy as np        # Numerical computing library
import pandas as pd       # Data manipulation and analysis library

# Print versions to ensure our environment is correctly loaded
print("NumPy version:", np.__version__)
print("pandas version:", pd.__version__)

NumPy version: 2.3.5
pandas version: 2.3.3


# 3. Dataset Provenance
- **Dataset Name:** Heart Failure Clinical Records
- **Source:** UCI Machine Learning Repository
- **Domain:** Healthcare / Clinical Cardiology
- **Target Variable:** `DEATH_EVENT` (0 = Survived, 1 = Deceased during follow-up period)

In [2]:
# ----------------------------------------------------
# DYNAMIC PATH RESOLUTION
# ----------------------------------------------------
# Identify the directory where this notebook is currently running
current_dir = Path.cwd()

# Handle folder depth dynamically:
# - If running from /notebooks/day05 -> move 2 levels up to project root
# - If running from /notebooks -> move 1 level up to project root
# - Otherwise, assume we are already at the project root
if current_dir.name == "day05":
    PROJECT_ROOT = current_dir.parent.parent
elif current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

# Define clean, relative paths for raw inputs, processed outputs, and schemas
RAW_PATH = PROJECT_ROOT / "data" / "day05" / "raw" / "heart_failure_clinical_records_dataset.csv"
CLEAN_PATH = PROJECT_ROOT / "data" / "day05" / "processed" / "heart_failure_cleaned.csv"
SCHEMA_CSV_PATH = PROJECT_ROOT / "output" / "day05" / "day05_cleaned_schema.csv"
SCHEMA_JSON_PATH = PROJECT_ROOT / "output" / "day05" / "day05_cleaned_schema.json"

# Automatically create destination directories if they don't exist yet (prevents FileNotFoundError)
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
SCHEMA_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

# Verify setup
print("Project Root:", PROJECT_ROOT)
print("Raw Path:", RAW_PATH)
print("Does the raw dataset exist on disk?", RAW_PATH.exists())

Project Root: C:\Users\hp\Desktop\AI_ENGINEERING\GitHub\ai-engineer-journey
Raw Path: C:\Users\hp\Desktop\AI_ENGINEERING\GitHub\ai-engineer-journey\data\day05\raw\heart_failure_clinical_records_dataset.csv
Does the raw dataset exist on disk? True


In [3]:
# ----------------------------------------------------
# LOAD DATA INTO PANDAS
# ----------------------------------------------------
# Read the raw CSV file into a pandas DataFrame memory structure
df = pd.read_csv(RAW_PATH)

# Verify the object type loaded is a pandas DataFrame
print("Loaded Data Type:", type(df))

Loaded Data Type: <class 'pandas.core.frame.DataFrame'>


In [4]:
# ----------------------------------------------------
# INITIAL DATA INSPECTION
# ----------------------------------------------------
# Inspect the first 5 rows to see what columns and values look like
print("--- FIRST 5 ROWS (HEAD) ---")
display(df.head())

# Inspect the last 5 rows to verify bottom rows aren't corrupted
print("\n--- LAST 5 ROWS (TAIL) ---")
display(df.tail())

# Print the dimensions: (total_rows, total_columns)
print("\nDataFrame Shape (rows, columns):", df.shape)

--- FIRST 5 ROWS (HEAD) ---


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1



--- LAST 5 ROWS (TAIL) ---


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
294,62.0,0,61,1,38,1,155000.0,1.1,143,1,1,270,0
295,55.0,0,1820,0,38,0,270000.0,1.2,139,0,0,271,0
296,45.0,0,2060,1,60,0,742000.0,0.8,138,0,0,278,0
297,45.0,0,2413,0,38,0,140000.0,1.4,140,1,1,280,0
298,50.0,0,196,0,45,0,395000.0,1.6,136,1,1,285,0



DataFrame Shape (rows, columns): (299, 13)


In [5]:
# ----------------------------------------------------
# STRUCTURAL DIAGNOSTICS
# ----------------------------------------------------
# View all raw column headers
print("--- COLUMN NAMES ---")
print(df.columns)

# Check data types inferred by pandas for each column
print("\n--- DATA TYPES ---")
print(df.dtypes)

# df.info() shows total rows, non-null counts, dtypes, and memory usage
print("\n--- DETAILED INFO ---")
df.info()

# df.describe() outputs summary statistics (mean, std, min, quartiles, max)
print("\n--- DESCRIPTIVE STATISTICS ---")
display(df.describe())

--- COLUMN NAMES ---
Index(['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes',
       'ejection_fraction', 'high_blood_pressure', 'platelets',
       'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time',
       'DEATH_EVENT'],
      dtype='object')

--- DATA TYPES ---
age                         float64
anaemia                       int64
creatinine_phosphokinase      int64
diabetes                      int64
ejection_fraction             int64
high_blood_pressure           int64
platelets                   float64
serum_creatinine            float64
serum_sodium                  int64
sex                           int64
smoking                       int64
time                          int64
DEATH_EVENT                   int64
dtype: object

--- DETAILED INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  ----

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
count,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000,299.00000,299.000000,299.000000,299.00000,299.000000,299.00000
mean,60.833893,0.431438,581.839465,0.418060,38.083612,0.351171,263358.029264,1.39388,136.625418,0.648829,0.32107,130.260870,0.32107
std,11.894809,0.496107,970.287881,0.494067,11.834841,0.478136,97804.236869,1.03451,4.412477,0.478136,0.46767,77.614208,0.46767
min,40.000000,0.000000,23.000000,0.000000,14.000000,0.000000,25100.000000,0.50000,113.000000,0.000000,0.00000,4.000000,0.00000
25%,51.000000,0.000000,116.500000,0.000000,30.000000,0.000000,212500.000000,0.90000,134.000000,0.000000,0.00000,73.000000,0.00000
50%,60.000000,0.000000,250.000000,0.000000,38.000000,0.000000,262000.000000,1.10000,137.000000,1.000000,0.00000,115.000000,0.00000
75%,70.000000,1.000000,582.000000,1.000000,45.000000,1.000000,303500.000000,1.40000,140.000000,1.000000,1.00000,203.000000,1.00000
max,95.000000,1.000000,7861.000000,1.000000,80.000000,1.000000,850000.000000,9.40000,148.000000,1.000000,1.00000,285.000000,1.00000


In [6]:
# ----------------------------------------------------
# STEP 1: MISSING-VALUE AUDIT & PRACTICE
# ----------------------------------------------------
# Count total missing (NaN) values per column
missing_counts = df.isna().sum()

# Calculate missing values as a percentage of total rows
missing_percent = df.isna().mean().mul(100).round(2)

print("Missing Counts per Column:\n", missing_counts)
print("\nMissing Percentages per Column:\n", missing_percent)

# --- PRACTICE ONLY: Learn how dropna() and fillna() work using a dummy Series ---
practice_values = pd.Series([42, np.nan, 57, np.nan, 35])
print("\n[PRACTICE] Is Null Mask:\n", practice_values.isna())
print("[PRACTICE] dropna() (Removes NaNs):\n", practice_values.dropna())
print("[PRACTICE] fillna(0) (Fills with 0):\n", practice_values.fillna(0))
print("[PRACTICE] fillna(median) (Fills with median 42):\n", practice_values.fillna(practice_values.median()))

Missing Counts per Column:
 age                         0
anaemia                     0
creatinine_phosphokinase    0
diabetes                    0
ejection_fraction           0
high_blood_pressure         0
platelets                   0
serum_creatinine            0
serum_sodium                0
sex                         0
smoking                     0
time                        0
DEATH_EVENT                 0
dtype: int64

Missing Percentages per Column:
 age                         0.0
anaemia                     0.0
creatinine_phosphokinase    0.0
diabetes                    0.0
ejection_fraction           0.0
high_blood_pressure         0.0
platelets                   0.0
serum_creatinine            0.0
serum_sodium                0.0
sex                         0.0
smoking                     0.0
time                        0.0
DEATH_EVENT                 0.0
dtype: float64

[PRACTICE] Is Null Mask:
 0    False
1     True
2    False
3     True
4    False
dtype: bool
[PRACTICE]

The original UCI dataset contained no missing values according to both the dataset metadata and my pandas missing-value audit.

In [7]:
# ----------------------------------------------------
# STEP 2: DUPLICATE ROW AUDIT
# ----------------------------------------------------
# df.duplicated() returns True for exact duplicate rows; .sum() counts them
duplicate_count = df.duplicated().sum()
print("Total duplicate rows found:", duplicate_count)

Total duplicate rows found: 0


In [8]:
# ----------------------------------------------------
# STEP 3: COLUMN NAME CLEANING & MUTATION SAFETY
# ----------------------------------------------------
# Print raw column strings using repr() to spot hidden spaces or special characters
print("Raw Column Names:")
for column in df.columns:
    print(repr(column))

# CRITICAL RULE: Make an explicit copy so we don't accidentally modify raw df
clean_df = df.copy()

# Standardize column headers: strip whitespace and convert to lowercase
# e.g., 'DEATH_EVENT' becomes 'death_event'
clean_df.columns = clean_df.columns.str.strip().str.lower()

print("\nCleaned Column Names:\n", clean_df.columns.tolist())

Raw Column Names:
'age'
'anaemia'
'creatinine_phosphokinase'
'diabetes'
'ejection_fraction'
'high_blood_pressure'
'platelets'
'serum_creatinine'
'serum_sodium'
'sex'
'smoking'
'time'
'DEATH_EVENT'

Cleaned Column Names:
 ['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time', 'death_event']


In [9]:
# ----------------------------------------------------
# STEP 4 & 5: DATA TYPE & BINARY VALUE AUDIT
# ----------------------------------------------------
print("--- CLEAN DF DATA TYPES ---")
print(clean_df.dtypes)

# Identify all binary (0 or 1) indicator columns
binary_columns = [
    "anaemia",
    "diabetes",
    "high_blood_pressure",
    "sex",
    "smoking",
    "death_event",
]

print("\n--- BINARY COLUMN VALUE DISTRIBUTION ---")
# Loop through binary columns to ensure they only contain valid 0s and 1s
for column in binary_columns:
    print(f"Distribution for '{column}':")
    print(clean_df[column].value_counts(dropna=False))
    print()

--- CLEAN DF DATA TYPES ---
age                         float64
anaemia                       int64
creatinine_phosphokinase      int64
diabetes                      int64
ejection_fraction             int64
high_blood_pressure           int64
platelets                   float64
serum_creatinine            float64
serum_sodium                  int64
sex                           int64
smoking                       int64
time                          int64
death_event                   int64
dtype: object

--- BINARY COLUMN VALUE DISTRIBUTION ---
Distribution for 'anaemia':
anaemia
0    170
1    129
Name: count, dtype: int64

Distribution for 'diabetes':
diabetes
0    174
1    125
Name: count, dtype: int64

Distribution for 'high_blood_pressure':
high_blood_pressure
0    194
1    105
Name: count, dtype: int64

Distribution for 'sex':
sex
1    194
0    105
Name: count, dtype: int64

Distribution for 'smoking':
smoking
0    203
1     96
Name: count, dtype: int64

Distribution for 'death_e

In [10]:
# ----------------------------------------------------
# STEP 6: DATA VALIDATION & RANGE CHECKS
# ----------------------------------------------------
# Verify min and max ranges for continuous clinical variables
# Note: Extreme values are valid clinical measurements, not errors!
print("Age Range:", clean_df["age"].min(), "to", clean_df["age"].max())
print("Ejection Fraction Range (%):", clean_df["ejection_fraction"].min(), "to", clean_df["ejection_fraction"].max())
print("Serum Creatinine Range (mg/dL):", clean_df["serum_creatinine"].min(), "to", clean_df["serum_creatinine"].max())

Age Range: 40.0 to 95.0
Ejection Fraction Range (%): 14 to 80
Serum Creatinine Range (mg/dL): 0.5 to 9.4


In [11]:
# ----------------------------------------------------
# STEP 7: NUMPY INTEGRATION (SERIES TO ARRAY)
# ----------------------------------------------------
# Extract column as a raw contiguous NumPy C-array in memory
age_array = clean_df["age"].to_numpy()
print("Type of extracted object:", type(age_array))

# Run fast NumPy vectorized math operations directly on the array
print("NumPy Mean Age:", np.mean(age_array))
print("NumPy Median Age:", np.median(age_array))
print("NumPy Standard Deviation:", np.std(age_array))
print("NumPy Percentiles (25th, 50th, 75th):", np.percentile(age_array, [25, 50, 75]))

# Verify that NumPy calculations match Pandas output
print("\nPandas describe() output for comparison:")
print(clean_df["age"].describe())

Type of extracted object: <class 'numpy.ndarray'>
NumPy Mean Age: 60.83389297658862
NumPy Median Age: 60.0
NumPy Standard Deviation: 11.874901429842655
NumPy Percentiles (25th, 50th, 75th): [51. 60. 70.]

Pandas describe() output for comparison:
count    299.000000
mean      60.833893
std       11.894809
min       40.000000
25%       51.000000
50%       60.000000
75%       70.000000
max       95.000000
Name: age, dtype: float64


In [12]:
# ----------------------------------------------------
# STEP 8: TARGET DISTRIBUTION ANALYSIS
# ----------------------------------------------------
# Count raw occurrences of survival (0) vs death (1)
death_counts = clean_df["death_event"].value_counts()

# Calculate percentage breakdown of target class distribution
# normalize=True gets fractions, mul(100) converts to %, round(2) cleans decimals
death_proportions = clean_df["death_event"].value_counts(normalize=True).mul(100).round(2)

print("Death Event Raw Counts:\n", death_counts)
print("\nDeath Event Observed Proportions (%):\n", death_proportions)

Death Event Raw Counts:
 death_event
0    203
1     96
Name: count, dtype: int64

Death Event Observed Proportions (%):
 death_event
0    67.89
1    32.11
Name: proportion, dtype: float64


In [13]:
# ----------------------------------------------------
# STEP 9: SPLIT-APPLY-COMBINE (GROUPBY AGGREGATIONS)
# ----------------------------------------------------
# Group data by death_event (0 vs 1) and calculate specific key metrics
group_summary = (
    clean_df
    .groupby("death_event")
    .agg(
        record_count=("age", "size"),                         # Total patients per group
        mean_age=("age", "mean"),                             # Average age
        median_ejection_fraction=("ejection_fraction", "median"), # Median heart pump efficiency
        mean_serum_creatinine=("serum_creatinine", "mean"),   # Average kidney efficiency marker
        median_followup_days=("time", "median"),              # Median days observed
    )
    .reset_index() # Converts index back into a standard column
)

display(group_summary)

,death_event,record_count,mean_age,median_ejection_fraction,mean_serum_creatinine,median_followup_days
0,0,203,58.761906,38.0,1.184877,172.0
1,1,96,65.215281,30.0,1.835833,44.5


In [14]:
# ----------------------------------------------------
# STEP 10: SORTING DATA
# ----------------------------------------------------
# Sort dataset by serum creatinine descending to inspect highest values
# (descriptive sorting, not a clinical risk ranking)
display(clean_df.sort_values(by="serum_creatinine", ascending=False).head(10))

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,death_event
9,80.0,1,123,0,35,1,388000.00,9.4,133,1,1,10,1
217,54.0,1,427,0,70,1,151000.00,9.0,137,0,0,196,1
52,60.0,0,3964,1,62,0,263358.03,6.8,146,0,0,43,1
131,60.0,1,1082,1,45,0,250000.00,6.1,131,1,0,107,0
28,58.0,1,60,0,38,0,153000.00,5.8,134,1,0,26,1
228,65.0,0,56,0,25,0,237000.00,5.0,130,0,0,207,0
48,80.0,1,553,0,20,1,140000.00,4.4,133,1,0,41,1
10,75.0,1,81,0,38,1,368000.00,4.0,131,1,1,10,1
282,42.0,0,64,0,30,0,215000.00,3.8,128,1,1,250,0
124,60.0,0,582,0,40,0,217000.00,3.7,134,1,0,96,1


In [15]:
# ----------------------------------------------------
# STEP 11: METADATA RELATIONAL JOIN (MERGE)
# ----------------------------------------------------
# 1. Create a secondary CSV metadata mapping file on disk
feature_dict_path = PROJECT_ROOT / "data" / "day05" / "feature_dictionary.csv"
feature_dict_path.parent.mkdir(parents=True, exist_ok=True)

csv_content = """feature,feature_group
age,demographic
anaemia,binary_condition
creatinine_phosphokinase,laboratory
diabetes,binary_condition
ejection_fraction,measurement
high_blood_pressure,binary_condition
platelets,laboratory
serum_creatinine,laboratory
serum_sodium,laboratory
sex,demographic
smoking,behavior
time,followup
death_event,target"""

with open(feature_dict_path, "w") as f:
    f.write(csv_content)

# 2. Read feature dictionary CSV back into a DataFrame
feature_dictionary = pd.read_csv(feature_dict_path)

# 3. Create descriptive statistics summary table
numeric_summary = (
    clean_df
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "feature"})
)

# 4. Perform Left Join on 'feature' column key
summary_with_metadata = numeric_summary.merge(
    feature_dictionary,
    on="feature",
    how="left",
)

print("Merged Table Shape:", summary_with_metadata.shape)
print("Unmatched Feature Categories (Should be 0):", summary_with_metadata["feature_group"].isna().sum())
display(summary_with_metadata)

Merged Table Shape: (13, 10)
Unmatched Feature Categories (Should be 0): 0


,feature,count,mean,std,min,25%,50%,75%,max,feature_group
0,age,299.0,60.833893,11.894809,40.0,51.0,60.0,70.0,95.0,demographic
1,anaemia,299.0,0.431438,0.496107,0.0,0.0,0.0,1.0,1.0,binary_condition
2,creatinine_phosphokinase,299.0,581.839465,970.287881,23.0,116.5,250.0,582.0,7861.0,laboratory
3,diabetes,299.0,0.418060,0.494067,0.0,0.0,0.0,1.0,1.0,binary_condition
4,ejection_fraction,299.0,38.083612,11.834841,14.0,30.0,38.0,45.0,80.0,measurement
5,high_blood_pressure,299.0,0.351171,0.478136,0.0,0.0,0.0,1.0,1.0,binary_condition
6,platelets,299.0,263358.029264,97804.236869,25100.0,212500.0,262000.0,303500.0,850000.0,laboratory
7,serum_creatinine,299.0,1.393880,1.034510,0.5,0.9,1.1,1.4,9.4,laboratory
8,serum_sodium,299.0,136.625418,4.412477,113.0,134.0,137.0,140.0,148.0,laboratory
9,sex,299.0,0.648829,0.478136,0.0,0.0,1.0,1.0,1.0,demographic


In [16]:
# ----------------------------------------------------
# STEP 12 & 13: DATASET EXPORT & INTEGRITY ASSERTIONS
# ----------------------------------------------------
# Save cleaned DataFrame to processed folder
clean_df.to_csv(CLEAN_PATH, index=False)
print("Cleaned file saved to disk. File exists?", CLEAN_PATH.exists())

# Re-read the file from disk to verify contents weren't altered during I/O
verification_df = pd.read_csv(CLEAN_PATH)

# Run strict programmatic assertions to guarantee file integrity
assert verification_df.shape == clean_df.shape, "Shape mismatch between memory and disk!"
assert list(verification_df.columns) == list(clean_df.columns), "Column mismatch between memory and disk!"

print("✅ Cleaned CSV verification passed successfully!")

Cleaned file saved to disk. File exists? True
✅ Cleaned CSV verification passed successfully!


In [17]:
# ----------------------------------------------------
# STEP 14: SCHEMA DEFINITION EXPORT (CSV + JSON)
# ----------------------------------------------------
# 1. Export Schema as CSV Table
schema_df = pd.DataFrame({
    "column_name": clean_df.columns,
    "dtype": [str(dtype) for dtype in clean_df.dtypes],
    "is_nullable": [bool(clean_df[col].isnull().any()) for col in clean_df.columns],
    "min_val": [clean_df[col].min() for col in clean_df.columns],
    "max_val": [clean_df[col].max() for col in clean_df.columns]
})
schema_df.to_csv(SCHEMA_CSV_PATH, index=False)

# 2. Export Schema as JSON Dictionary (great for downstream ML validation pipelines)
schema_dict = {
    col: {
        "dtype": str(clean_df[col].dtype),
        "is_nullable": bool(clean_df[col].isnull().any()),
        "min_val": float(clean_df[col].min()),
        "max_val": float(clean_df[col].max())
    }
    for col in clean_df.columns
}
with open(SCHEMA_JSON_PATH, "w") as f:
    json.dump(schema_dict, f, indent=4)

print(f"✅ Schema CSV exported to: {SCHEMA_CSV_PATH}")
print(f"✅ Schema JSON exported to: {SCHEMA_JSON_PATH}")

✅ Schema CSV exported to: C:\Users\hp\Desktop\AI_ENGINEERING\GitHub\ai-engineer-journey\output\day05\day05_cleaned_schema.csv
✅ Schema JSON exported to: C:\Users\hp\Desktop\AI_ENGINEERING\GitHub\ai-engineer-journey\output\day05\day05_cleaned_schema.json


# 15. Key Findings
1. **Strongest Predictors:** `time` (follow-up period length) has the strongest negative correlation with death events. Patients who survived longer follow-up windows had lower fatality rates.
2. **Clinical Markers:** Higher `serum_creatinine` and lower `ejection_fraction` are strongly associated with higher mortality rates.
3. **Class Balance:** The target variable (`death_event`) exhibits a ~67.9% survival vs ~32.1% mortality distribution.

# 16. Limitations
1. **Sample Size:** The dataset contains 299 records, which is small for complex statistical modelling without cross-validation or regularization.
2. **Observational Scope:** This EDA represents purely descriptive correlation analysis and cannot be used to infer causal medical relationships.

### Step-by-Step Code to Add to  Jupyter Notebook
Here are the exact cells to run at the end of notebooks/day05_eda.ipynb notebook to fulfill all these requirements:

In [18]:
### Cell A — Code (Create Cleaned CSV Schema & Output EDA Summary)
### Add and run this code cell right after  dataset cleaning and merge steps:


# ----------------------------------------------------
# 1. CREATE CLEANED CSV SCHEMA
# ----------------------------------------------------
schema_rows = []

for column in clean_df.columns:
    schema_rows.append(
        {
            "column": column,
            "dtype": str(clean_df[column].dtype),
            "missing_count": int(clean_df[column].isna().sum()),
            "unique_count": int(clean_df[column].nunique(dropna=True)),
        }
    )

schema_df = pd.DataFrame(schema_rows)

print("--- CLEANED SCHEMA PROOF ---")
display(schema_df)

# Save schema CSV
schema_df.to_csv(SCHEMA_CSV_PATH, index=False)
print(f"✅ Cleaned schema exported to: {SCHEMA_CSV_PATH}\n")


# ----------------------------------------------------
# 2. CREATE AND EXPORT EDA SUMMARY
# ----------------------------------------------------
eda_summary = clean_df[
    [
        "age",
        "ejection_fraction",
        "serum_creatinine",
        "serum_sodium",
        "time",
    ]
].describe().T

EDA_PATH = PROJECT_ROOT / "output" / "day05_eda_summary.csv"
EDA_PATH.parent.mkdir(parents=True, exist_ok=True)

eda_summary.to_csv(EDA_PATH)
print(f"✅ EDA summary exported to: {EDA_PATH}")
display(eda_summary)

--- CLEANED SCHEMA PROOF ---


,column,dtype,missing_count,unique_count
0,age,float64,0,47
1,anaemia,int64,0,2
2,creatinine_phosphokinase,int64,0,208
3,diabetes,int64,0,2
4,ejection_fraction,int64,0,17
5,high_blood_pressure,int64,0,2
6,platelets,float64,0,176
7,serum_creatinine,float64,0,40
8,serum_sodium,int64,0,27
9,sex,int64,0,2


✅ Cleaned schema exported to: C:\Users\hp\Desktop\AI_ENGINEERING\GitHub\ai-engineer-journey\output\day05\day05_cleaned_schema.csv

✅ EDA summary exported to: C:\Users\hp\Desktop\AI_ENGINEERING\GitHub\ai-engineer-journey\output\day05_eda_summary.csv


,count,mean,std,min,25%,50%,75%,max
age,299.0,60.833893,11.894809,40.0,51.0,60.0,70.0,95.0
ejection_fraction,299.0,38.083612,11.834841,14.0,30.0,38.0,45.0,80.0
serum_creatinine,299.0,1.393880,1.034510,0.5,0.9,1.1,1.4,9.4
serum_sodium,299.0,136.625418,4.412477,113.0,134.0,137.0,140.0,148.0
time,299.0,130.260870,77.614208,4.0,73.0,115.0,203.0,285.0


In [19]:
### Cell B — Code (Calculate Exact Metrics for Blanks)
### Run this helper code cell to print the exact numbers needed to fill in markdown section:
# Helper block to print exact values for your markdown report


rows, cols = clean_df.shape
missing_total = clean_df.isna().sum().sum()
duplicates_total = clean_df.duplicated().sum()
death_dist = clean_df["death_event"].value_counts(normalize=True).mul(100).round(2).to_dict()
med_age = clean_df["age"].median()
med_ef = clean_df["ejection_fraction"].median()
med_sc = clean_df["serum_creatinine"].median()

print(f"1. Rows: {rows}, Columns: {cols}")
print(f"2. Missing values: {missing_total}")
print(f"3. Duplicates: {duplicates_total}")
print(f"4. Death-event distribution: {death_dist[0]}% survived (0), {death_dist[1]}% deceased (1)")
print(f"5. Median age: {med_age}")
print(f"6. Median ejection fraction: {med_ef}")
print(f"7. Median serum creatinine: {med_sc}")

1. Rows: 299, Columns: 13
2. Missing values: 0
3. Duplicates: 0
4. Death-event distribution: 67.89% survived (0), 32.11% deceased (1)
5. Median age: 60.0
6. Median ejection fraction: 38.0
7. Median serum creatinine: 1.1


### Cell C — Markdown (Key Observations — Filled with Calculated Values)
### Create a Markdown cell at the bottom of notebook and fill in values printed from Cell B:
## Key observations


1. Dataset contains **299** rows and **13** columns.
2. Missing-value audit found **0** missing values.
3. Duplicate-row audit found **0** duplicates.
4. Death-event distribution was **67.89% survived (0) and 32.11% deceased (1)**.
5. Median age was **60.0**.
6. Median ejection fraction was **38.0**.
7. Median serum creatinine was **1.1**.
8. Grouped descriptive statistics differed between death-event categories for several variables.

These observations are descriptive and should not be interpreted as causal or clinical conclusions.

## Limitations

- This is a small historical dataset.
- The analysis is descriptive.
- No causal inference is performed.
- No predictive model is trained on Day 5.
- No external validation has been performed.
- Observed associations must not be treated as clinical recommendations.

## Data source

Heart Failure Clinical Records, UCI Machine Learning Repository.

DOI: 10.24432/C5Z89R

The dataset is distributed under the Creative Commons Attribution 4.0 license.